# ETHIX — AI Fairness & Bias Evaluation
### Loan Approval Dataset · End-to-End Data Science Pipeline

---

This notebook walks through the full ETHIX pipeline on the **Loan Approval dataset**.
The goal isn't just to build an accurate model — it's to ask whether the model is *fair*.

Specifically: does a person's education level (Graduate vs Not Graduate) influence
their chances of getting a loan approved, independent of their actual financial profile?

**Pipeline overview:**
1. Load and inspect the data
2. Exploratory Data Analysis
3. Preprocessing
4. Model training (Logistic Regression)
5. Performance evaluation
6. Fairness metrics — SPD, DIR, EOD, Individual Fairness
7. Bias mitigation experiment
8. Ethical Score

## 0. Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    roc_auc_score, roc_curve
)

warnings.filterwarnings('ignore')

# consistent plots
plt.rcParams.update({
    'figure.facecolor': '#1a1a2e',
    'axes.facecolor':   '#16213e',
    'axes.edgecolor':   '#3a3a5c',
    'axes.labelcolor':  '#e0e0e0',
    'xtick.color':      '#e0e0e0',
    'ytick.color':      '#e0e0e0',
    'text.color':       '#e0e0e0',
    'grid.color':       '#3a3a5c',
    'grid.alpha':        0.4,
})

PALETTE = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
print('Libraries loaded.')

Libraries loaded.


## 1. Load the Dataset

Using the Loan Approval dataset. Contains ~4,200 loan applications with financial
features and a binary outcome (Approved / Rejected).

First thing I always do — check shape, dtypes, and whether there are obvious issues
like a stray ID column or weird nulls.

In [2]:
df = pd.read_csv('../data/loan_approval_dataset.csv')

# clean column names right away — spaces and mixed case are annoying
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
    .str.replace(r'[^\w]', '_', regex=True)
)

print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

Shape: (4269, 13)
Columns: ['loan_id', 'no_of_dependents', 'education', 'self_employed', 'income_annum', 'loan_amount', 'loan_term', 'cibil_score', 'residential_assets_value', 'commercial_assets_value', 'luxury_assets_value', 'bank_asset_value', 'loan_status']


,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,1,2,Graduate,No,9600000,29900000,12,778,2400000,17600000,22700000,8000000,Approved
1,2,0,Not Graduate,Yes,4100000,12200000,8,417,2700000,2200000,8800000,3300000,Rejected
2,3,3,Graduate,No,9100000,29700000,20,506,7100000,4500000,33300000,12800000,Rejected
3,4,3,Graduate,No,8200000,30700000,8,467,18200000,3300000,23300000,7900000,Rejected
4,5,5,Not Graduate,Yes,9800000,24200000,20,382,12400000,8200000,29400000,5000000,Rejected


In [3]:
# quick sanity check
print('--- Dtypes ---')
print(df.dtypes)
print()
print('--- Missing values ---')
print(df.isnull().sum())
print()
print('--- Target distribution ---')
print(df['loan_status'].value_counts())

--- Dtypes ---
loan_id                      int64
no_of_dependents             int64
education                   object
self_employed               object
income_annum                 int64
loan_amount                  int64
loan_term                    int64
cibil_score                  int64
residential_assets_value     int64
commercial_assets_value      int64
luxury_assets_value          int64
bank_asset_value             int64
loan_status                 object
dtype: object

--- Missing values ---
loan_id                     0
no_of_dependents            0
education                   0
self_employed               0
income_annum                0
loan_amount                 0
loan_term                   0
cibil_score                 0
residential_assets_value    0
commercial_assets_value     0
luxury_assets_value         0
bank_asset_value            0
loan_status                 0
dtype: int64

--- Target distribution ---
loan_status
Approved    2656
Rejected    1613
Name: count, d

In [4]:
# loan_id is pure identifier — every value is unique, zero predictive value
# drop it before anything else
assert df['loan_id'].nunique() == len(df), 'loan_id is not fully unique??'
df.drop(columns=['loan_id'], inplace=True)
print('loan_id dropped. Remaining columns:', df.columns.tolist())

loan_id dropped. Remaining columns: ['no_of_dependents', 'education', 'self_employed', 'income_annum', 'loan_amount', 'loan_term', 'cibil_score', 'residential_assets_value', 'commercial_assets_value', 'luxury_assets_value', 'bank_asset_value', 'loan_status']


In [5]:
df.describe()

,no_of_dependents,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value
count,4269.000000,4.269000e+03,4.269000e+03,4269.000000,4269.000000,4.269000e+03,4.269000e+03,4.269000e+03,4.269000e+03
mean,2.498712,5.059124e+06,1.513345e+07,10.900445,599.936051,7.472617e+06,4.973155e+06,1.512631e+07,4.976692e+06
std,1.695910,2.806840e+06,9.043363e+06,5.709187,172.430401,6.503637e+06,4.388966e+06,9.103754e+06,3.250185e+06
min,0.000000,2.000000e+05,3.000000e+05,2.000000,300.000000,-1.000000e+05,0.000000e+00,3.000000e+05,0.000000e+00
25%,1.000000,2.700000e+06,7.700000e+06,6.000000,453.000000,2.200000e+06,1.300000e+06,7.500000e+06,2.300000e+06
50%,3.000000,5.100000e+06,1.450000e+07,10.000000,600.000000,5.600000e+06,3.700000e+06,1.460000e+07,4.600000e+06
75%,4.000000,7.500000e+06,2.150000e+07,16.000000,748.000000,1.130000e+07,7.600000e+06,2.170000e+07,7.100000e+06
max,5.000000,9.900000e+06,3.950000e+07,20.000000,900.000000,2.910000e+07,1.940000e+07,3.920000e+07,1.470000e+07


## 2. Exploratory Data Analysis

Before touching the model I want to understand the data visually.
Key questions:
- Is the target balanced?
- Does education already show a bias pattern in the raw data?
- Which numerical features actually separate the two classes?
- Are there correlations worth knowing about?

### 2.1 Target Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

counts = df['loan_status'].str.strip().value_counts()
bars = ax.bar(counts.index, counts.values,
              color=['#4C72B0', '#DD8452'], edgecolor='white', linewidth=0.8)

for bar, val in zip(bars, counts.values):
    pct = val / len(df) * 100
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 30,
            f'{val:,}\n({pct:.1f}%)',
            ha='center', fontsize=10, fontweight='bold')

ax.set_title('Loan Status Distribution', fontsize=13, pad=10)
ax.set_ylabel('Count')
ax.margins(y=0.15)
plt.tight_layout()
plt.show()

# mild class imbalance — not bad enough to need SMOTE or reweighting

### 2.2 Education vs Loan Status — the key fairness question

In [ ]:
# this is the chart I care about most before training anything
# if Non-Graduates are already being rejected more in the raw data,
# the model will almost certainly learn that pattern

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df_plot = df.copy()
df_plot['education']    = df_plot['education'].str.strip()
df_plot['loan_status']  = df_plot['loan_status'].str.strip()

# absolute counts
sns.countplot(data=df_plot, x='education', hue='loan_status',
              palette=['#4C72B0','#DD8452'], ax=axes[0],
              edgecolor='white', linewidth=0.5)
axes[0].set_title('Education vs Loan Status (counts)', fontsize=11)
axes[0].set_xlabel('Education')
axes[0].set_ylabel('Count')
legend = axes[0].get_legend()
legend.get_frame().set_facecolor('#1a1a2e')

# approval rates — more honest than raw counts
rates = (df_plot.groupby('education')['loan_status']
         .apply(lambda x: (x == 'Approved').mean())
         .reset_index())
rates.columns = ['education', 'approval_rate']

bars = axes[1].bar(rates['education'], rates['approval_rate'],
                   color=['#4C72B0','#DD8452'], edgecolor='white', linewidth=0.8)
for bar, val in zip(bars, rates['approval_rate']):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.01,
                 f'{val:.1%}', ha='center', fontsize=11, fontweight='bold')
axes[1].set_title('Approval Rate by Education', fontsize=11)
axes[1].set_ylabel('Approval Rate')
axes[1].set_ylim(0, 1.1)

plt.tight_layout()
plt.show()

print('Approval rates:')
print(rates.to_string(index=False))

### 2.3 Numerical Features — how do they differ across classes?

In [ ]:
num_cols = ['no_of_dependents', 'income_annum', 'loan_amount',
            'loan_term', 'cibil_score', 'residential_assets_value',
            'commercial_assets_value', 'luxury_assets_value', 'bank_asset_value']

df_plot = df.copy()
df_plot['loan_status'] = df_plot['loan_status'].str.strip()

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    classes = df_plot['loan_status'].unique()
    data = [df_plot.loc[df_plot['loan_status'] == c, col].dropna().values
            for c in classes]

    bp = axes[i].boxplot(data, labels=classes, patch_artist=True,
                          medianprops=dict(color='white', linewidth=2))
    for patch, color in zip(bp['boxes'], ['#4C72B0', '#DD8452']):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)

    axes[i].set_title(col, fontsize=9)
    axes[i].tick_params(labelsize=8)

plt.suptitle('Numerical Features by Loan Status', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# cibil_score shows the clearest separation — no surprise,
# it is literally designed to predict creditworthiness

### 2.4 Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))  # upper triangle only

sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, square=True,
            linewidths=0.5, linecolor='#3a3a5c',
            annot_kws={'size': 8}, ax=ax)

ax.set_title('Correlation Matrix — Numerical Features', fontsize=12, pad=10)
plt.tight_layout()
plt.show()

# income_annum and loan_amount are moderately correlated — expected
# asset values cluster together as well

## 3. Preprocessing

Nothing fancy here — standard steps.
The main thing to be careful about is **not leaking the sensitive attribute** into
the scaler or encoder in a way that would distort the fairness analysis later.

In [ ]:
df_clean = df.copy()

# strip whitespace from string columns — the CSV has leading spaces
for col in df_clean.select_dtypes(include='object').columns:
    df_clean[col] = df_clean[col].str.strip()

TARGET    = 'loan_status'
SENSITIVE = 'education'

print('Target classes:  ', df_clean[TARGET].unique())
print('Sensitive groups:', df_clean[SENSITIVE].unique())

In [ ]:
# missing values
print('Missing before:', df_clean.isnull().sum().sum())

for col in num_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

cat_cols = ['education', 'self_employed']
for col in cat_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

print('Missing after: ', df_clean.isnull().sum().sum())

In [ ]:
# encode categoricals
# keeping a record of the encoders so we can decode labels for charts later
encoders = {}

for col in cat_cols + [TARGET]:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))
    encoders[col] = le
    print(f'{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}')

In [ ]:
# scale numerics — exclude target and sensitive attribute
cols_to_scale = [c for c in num_cols
                 if c in df_clean.columns
                 and c != TARGET and c != SENSITIVE]

scaler = StandardScaler()
df_clean[cols_to_scale] = scaler.fit_transform(df_clean[cols_to_scale])
print('Scaled columns:', cols_to_scale)

In [ ]:
# train / test split
# stratify keeps the class ratio the same in both splits

feature_cols = [c for c in df_clean.columns if c != TARGET]

X = df_clean[feature_cols]
y = df_clean[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# pull out sensitive column from test set for fairness evaluation
sensitive_test = X_test[SENSITIVE].reset_index(drop=True)

print(f'Train: {X_train.shape}   Test: {X_test.shape}')
print(f'Train target ratio: {y_train.mean():.3f}')
print(f'Test  target ratio: {y_test.mean():.3f}')

## 4. Model Training

Using Logistic Regression as the baseline. Main reasons:
- Interpretable coefficients — I can see exactly what the model learned
- Fast to train and iterate on
- Good baseline before trying anything more complex

max_iter=1000 to avoid convergence warnings on this size dataset.

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)
print('Model trained.')
print(f'Classes: {model.classes_}')

In [ ]:
# feature importance — absolute coefficient magnitude
coefs      = pd.Series(np.abs(model.coef_[0]), index=X_train.columns)
coefs_top  = coefs.nlargest(10).sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(coefs_top.index, coefs_top.values,
               color='#4C72B0', edgecolor='white', linewidth=0.5)
ax.set_xlabel('|Coefficient|')
ax.set_title('Top 10 Features by Coefficient Magnitude', fontsize=12, pad=8)
plt.tight_layout()
plt.show()

# cibil_score should dominate — it is the single best predictor of loan repayment
# if education is near the top, that is a fairness red flag

## 5. Performance Evaluation

Standard classification metrics on the held-out test set.
Always evaluate on data the model has never seen — anything else is cheating.

In [ ]:
y_pred      = model.predict(X_test)
y_pred_prob = model.predict_proba(X_test)[:, 1]

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
rec  = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1   = f1_score(y_test, y_pred, average='weighted', zero_division=0)
auc  = roc_auc_score(y_test, y_pred_prob)

metrics_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC'],
    'Score':  [acc, prec, rec, f1, auc]
})
metrics_df['Score'] = metrics_df['Score'].round(4)
print(metrics_df.to_string(index=False))

In [ ]:
# confusion matrix + ROC side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# -- confusion matrix --
cm = confusion_matrix(y_test, y_pred)
labels = encoders[TARGET].classes_
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels,
            ax=axes[0], linewidths=0.5,
            annot_kws={'size': 13, 'weight': 'bold'})
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix', fontsize=12)

# -- ROC curve --
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
axes[1].plot(fpr, tpr, color='#4C72B0', lw=2,
             label=f'ROC (AUC = {auc:.3f})')
axes[1].plot([0,1],[0,1], color='#555555', lw=1, linestyle='--', label='Random')
axes[1].fill_between(fpr, tpr, alpha=0.12, color='#4C72B0')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve', fontsize=12)
legend = axes[1].legend(loc='lower right')
legend.get_frame().set_facecolor('#1a1a2e')

plt.tight_layout()
plt.show()

In [ ]:
print(classification_report(y_test, y_pred,
      target_names=encoders[TARGET].classes_,
      zero_division=0))

## 6. Fairness Evaluation

This is where things get interesting. A 92% accurate model can still be
deeply unfair to a subgroup.

**Sensitive attribute:** `education` (Graduate vs Not Graduate)

Three group-fairness metrics:
- **SPD** — are approval rates equal across groups?
- **DIR** — what is the ratio of approval rates? (80% rule)
- **EOD** — when someone *deserves* approval, does the model catch it equally?

Plus individual fairness — do similar people get similar decisions?

In [ ]:
# split test predictions by education group
y_pred_arr = np.array(y_pred, dtype=float)
y_test_arr = np.array(y_test, dtype=float)
sens_arr   = np.array(sensitive_test)

groups = np.unique(sens_arr)
group_labels = {g: encoders[SENSITIVE].inverse_transform([int(g)])[0] for g in groups}

# positive (approved) prediction rate per group
rates = {}
for g in groups:
    mask = (sens_arr == g)
    rates[group_labels[g]] = y_pred_arr[mask].mean()

print('Approval rates by education group:')
for name, rate in rates.items():
    print(f'  {name}: {rate:.4f} ({rate:.1%})')

### 6.1 Statistical Parity Difference (SPD)

In [ ]:
# SPD = P(approved | unprivileged) - P(approved | privileged)
# privileged = majority group (most frequent in dataset)

sorted_groups = sorted(rates.items(), key=lambda x: -x[1])
privileged_name,   rate_priv   = sorted_groups[0]
unprivileged_name, rate_unpriv = sorted_groups[1]

spd = rate_unpriv - rate_priv

print(f'Privileged group   ({privileged_name}):   {rate_priv:.4f}')
print(f'Unprivileged group ({unprivileged_name}): {rate_unpriv:.4f}')
print(f'SPD = {spd:.4f}')
print()
if abs(spd) <= 0.1:
    print('Result: FAIR  (|SPD| <= 0.1)')
else:
    print(f'Result: UNFAIR  — difference of {abs(spd):.1%} between groups')

### 6.2 Disparate Impact Ratio (DIR)

In [ ]:
# DIR = P(approved | unprivileged) / P(approved | privileged)
# The "80% rule" — DIR should be >= 0.8 to be considered fair
# This comes from US employment law (EEOC guidelines)

dir_value = rate_unpriv / rate_priv if rate_priv > 0 else 0

print(f'DIR = {rate_unpriv:.4f} / {rate_priv:.4f} = {dir_value:.4f}')
print()
if dir_value >= 0.8:
    print(f'Result: FAIR  (DIR >= 0.8, the 80% rule)')
else:
    print(f'Result: UNFAIR  — {unprivileged_name} only gets {dir_value:.1%} '
          f'of the approval chances of {privileged_name}')

### 6.3 Equal Opportunity Difference (EOD)

In [ ]:
# EOD = TPR(unprivileged) - TPR(privileged)
# TPR = of all people who DESERVE approval, how many did we actually approve?
# this is stricter than SPD — it controls for the actual ground truth

def tpr_for_group(mask):
    positives = (y_test_arr[mask] == 1)
    if positives.sum() == 0:
        return 0.0
    return float((y_pred_arr[mask][positives] == 1).sum() / positives.sum())

# rebuild masks using group labels
priv_encoded   = encoders[SENSITIVE].transform([privileged_name])[0]
unpriv_encoded = encoders[SENSITIVE].transform([unprivileged_name])[0]

mask_priv   = (sens_arr == priv_encoded)
mask_unpriv = (sens_arr == unpriv_encoded)

tpr_priv   = tpr_for_group(mask_priv)
tpr_unpriv = tpr_for_group(mask_unpriv)
eod        = tpr_unpriv - tpr_priv

print(f'TPR ({privileged_name}):   {tpr_priv:.4f}')
print(f'TPR ({unprivileged_name}): {tpr_unpriv:.4f}')
print(f'EOD = {eod:.4f}')
print()
if abs(eod) <= 0.1:
    print('Result: FAIR  (|EOD| <= 0.1)')
else:
    print(f'Result: UNFAIR  — model misses deserving {unprivileged_name} '
          f'applicants {abs(eod):.1%} more often')

### 6.4 Individual Fairness

In [ ]:
# randomly sample pairs of people from the test set
# check whether 'similar' people (close in feature space) get the same prediction
# this is a heuristic — not a rigorous proof — but it is fast and readable

np.random.seed(42)
X_test_arr = X_test.values.astype(float)
n          = len(X_test_arr)
n_pairs    = 500
threshold  = 0.3

idx_a = np.random.randint(0, n, size=n_pairs)
idx_b = np.random.randint(0, n, size=n_pairs)
# make sure we never compare a person to themselves
same  = (idx_a == idx_b)
idx_b[same] = (idx_b[same] + 1) % n

dists         = np.linalg.norm(X_test_arr[idx_a] - X_test_arr[idx_b], axis=1)
similar_mask  = dists < threshold
inconsistent  = int((y_pred_arr[idx_a][similar_mask] != y_pred_arr[idx_b][similar_mask]).sum())
total_similar = int(similar_mask.sum())
ind_score     = 1.0 - (inconsistent / total_similar) if total_similar > 0 else 1.0

print(f'Pairs checked:               {n_pairs}')
print(f'Similar pairs (dist < {threshold}): {total_similar}')
print(f'Inconsistent predictions:    {inconsistent}')
print(f'Individual Fairness Score:   {ind_score:.4f}')
print()
print('Result:', 'FAIR (>= 0.8)' if ind_score >= 0.8 else 'UNFAIR (< 0.8)')

### 6.5 Fairness Summary

In [ ]:
# put it all together in one table
summary = pd.DataFrame([
    {'Metric': 'Statistical Parity Difference (SPD)',
     'Value': round(spd, 4),
     'Fair Range': '−0.1 to +0.1',
     'Result': 'FAIR' if abs(spd) <= 0.1 else 'UNFAIR'},

    {'Metric': 'Disparate Impact Ratio (DIR)',
     'Value': round(dir_value, 4),
     'Fair Range': '>= 0.8',
     'Result': 'FAIR' if dir_value >= 0.8 else 'UNFAIR'},

    {'Metric': 'Equal Opportunity Difference (EOD)',
     'Value': round(eod, 4),
     'Fair Range': '−0.1 to +0.1',
     'Result': 'FAIR' if abs(eod) <= 0.1 else 'UNFAIR'},

    {'Metric': 'Individual Fairness Score',
     'Value': round(ind_score, 4),
     'Fair Range': '>= 0.8',
     'Result': 'FAIR' if ind_score >= 0.8 else 'UNFAIR'},
])

print(summary.to_string(index=False))

In [ ]:
# visual summary — fairness dashboard
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
fig.suptitle('Fairness Metrics — Education (Graduate vs Not Graduate)',
             fontsize=13, y=1.03)

metric_info = [
    ('SPD',  spd,       (-0.1, 0.1),  'Statistical\nParity Diff.'),
    ('DIR',  dir_value, (0.8,  1.2),  'Disparate\nImpact Ratio'),
    ('EOD',  eod,       (-0.1, 0.1),  'Equal\nOpportunity Diff.'),
    ('IFS',  ind_score, (0.8,  1.0),  'Individual\nFairness Score'),
]

for ax, (label, value, fair_range, title) in zip(axes, metric_info):
    lo, hi  = fair_range
    is_fair = lo <= value <= hi
    color   = '#55A868' if is_fair else '#C44E52'

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')

    ax.text(0.5, 0.82, title,    ha='center', va='center',
            fontsize=10, fontweight='bold', transform=ax.transAxes)
    ax.text(0.5, 0.55, f'{value:.3f}', ha='center', va='center',
            fontsize=26, fontweight='bold', color=color, transform=ax.transAxes)
    ax.text(0.5, 0.32, f'Fair: {lo} to {hi}', ha='center', va='center',
            fontsize=8, color='#aaaaaa', transform=ax.transAxes)

    verdict = 'FAIR ✓' if is_fair else 'UNFAIR ✗'
    ax.text(0.5, 0.12, verdict, ha='center', va='center',
            fontsize=11, fontweight='bold', color=color, transform=ax.transAxes,
            bbox=dict(boxstyle='round,pad=0.3',
                      facecolor=color+'33', edgecolor=color))

    ax.set_facecolor('#16213e')

plt.tight_layout()
plt.show()

## 7. Bias Mitigation

Simplest mitigation strategy: retrain the model **without** the sensitive attribute.

The idea is that if `education` is not in the feature set, the model cannot
directly use it to discriminate. There is still a risk of proxy discrimination
(e.g. income correlates with education) but it is a good starting point.

Trade-off to watch: accuracy vs fairness. Removing a predictive feature
usually costs a few points of accuracy.

In [ ]:
# retrain without the education column
feature_cols_miti = [c for c in feature_cols if c != SENSITIVE]

X_train_m = X_train[feature_cols_miti]
X_test_m  = X_test[feature_cols_miti]

model_miti = LogisticRegression(max_iter=1000, random_state=42)
model_miti.fit(X_train_m, y_train)

y_pred_m = model_miti.predict(X_test_m)
acc_miti = accuracy_score(y_test, y_pred_m)

print(f'Original model accuracy:  {acc:.4f}')
print(f'Mitigated model accuracy: {acc_miti:.4f}')
print(f'Accuracy change:          {acc_miti - acc:+.4f}')

In [ ]:
# fairness metrics for mitigated model
y_pred_m_arr = np.array(y_pred_m, dtype=float)

rates_m = {}
for g, name in group_labels.items():
    mask = (sens_arr == g)
    rates_m[name] = y_pred_m_arr[mask].mean()

rate_priv_m   = rates_m[privileged_name]
rate_unpriv_m = rates_m[unprivileged_name]

spd_m      = rate_unpriv_m - rate_priv_m
dir_m      = rate_unpriv_m / rate_priv_m if rate_priv_m > 0 else 0
tpr_priv_m = tpr_for_group(mask_priv)

# need to recalculate tpr_for_group with mitigated predictions
def tpr_group_m(mask):
    pos = (y_test_arr[mask] == 1)
    if pos.sum() == 0: return 0.0
    return float((y_pred_m_arr[mask][pos] == 1).sum() / pos.sum())

eod_m = tpr_group_m(mask_unpriv) - tpr_group_m(mask_priv)

print('Fairness comparison — Original vs Mitigated:')
print(f'  SPD: {spd:.4f}  →  {spd_m:.4f}')
print(f'  DIR: {dir_value:.4f}  →  {dir_m:.4f}')
print(f'  EOD: {eod:.4f}  →  {eod_m:.4f}')

In [ ]:
# comparison chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# accuracy comparison
ax = axes[0]
vals   = [acc, acc_miti]
labels = ['Original', 'Mitigated']
bars = ax.bar(labels, vals, color=['#4C72B0','#55A868'],
              edgecolor='white', linewidth=0.8, width=0.5)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.003,
            f'{v:.3f}', ha='center', fontsize=12, fontweight='bold')
ax.set_title('Accuracy Comparison', fontsize=11)
ax.set_ylim(0, 1.1)

# fairness metrics comparison
ax = axes[1]
metric_names = ['SPD (abs)', 'DIR', 'EOD (abs)']
orig_vals_f  = [abs(spd),    dir_value,  abs(eod)]
miti_vals_f  = [abs(spd_m),  dir_m,      abs(eod_m)]

x     = np.arange(len(metric_names))
width = 0.35

ax.bar(x - width/2, orig_vals_f, width, label='Original',
       color='#4C72B0', edgecolor='white', linewidth=0.5)
ax.bar(x + width/2, miti_vals_f, width, label='Mitigated',
       color='#55A868', edgecolor='white', linewidth=0.5)

ax.set_xticks(x)
ax.set_xticklabels(metric_names)
ax.set_title('Fairness Metrics Comparison', fontsize=11)
legend = ax.legend()
legend.get_frame().set_facecolor('#1a1a2e')

plt.tight_layout()
plt.show()

## 8. Ethical Score

Combining accuracy and fairness into a single composite score.

The weights are somewhat subjective — I gave accuracy 35% and split the
remaining 65% across the three fairness metrics weighted by how actionable
they are:

```
Ethical Score = 0.35 × Accuracy
              + 0.25 × SPD score
              + 0.25 × DIR score
              + 0.15 × EOD score
```

Each fairness metric is converted to a 0–1 goodness score before combining.

In [ ]:
def spd_to_score(v): return max(0.0, 1.0 - abs(v) / 0.5)

def dir_to_score(v):
    if v >= 1.0:   return min(1.0, 0.8 + v * 0.2)
    elif v >= 0.8: return 0.6 + (v - 0.8) / 0.2 * 0.4
    else:          return max(0.0, v / 0.8 * 0.6)

def eod_to_score(v): return max(0.0, 1.0 - abs(v) / 0.5)

def ethical_score(accuracy, spd_val, dir_val, eod_val, weights=None):
    if weights is None:
        weights = {'accuracy': 0.35, 'spd': 0.25, 'dir': 0.25, 'eod': 0.15}

    score = (
        weights['accuracy'] * accuracy
        + weights['spd']    * spd_to_score(spd_val)
        + weights['dir']    * dir_to_score(dir_val)
        + weights['eod']    * eod_to_score(eod_val)
    )
    return round(score, 4)


es_orig = ethical_score(acc,      spd,   dir_value, eod)
es_miti = ethical_score(acc_miti, spd_m, dir_m,     eod_m)

def grade(s):
    if s >= 0.85: return 'Excellent'
    if s >= 0.70: return 'Good'
    if s >= 0.55: return 'Fair'
    if s >= 0.40: return 'Poor'
    return 'Critical'

print(f'Original  model — Ethical Score: {es_orig:.4f}  ({grade(es_orig)})')
print(f'Mitigated model — Ethical Score: {es_miti:.4f}  ({grade(es_miti)})')
print(f'Improvement: {es_miti - es_orig:+.4f}')

In [ ]:
# gauge chart for original model
def draw_gauge(score, title, ax):
    if score >= 0.85:   color = '#2ecc71'
    elif score >= 0.70: color = '#27ae60'
    elif score >= 0.55: color = '#f39c12'
    elif score >= 0.40: color = '#e67e22'
    else:               color = '#e74c3c'

    theta_bg  = np.linspace(np.pi, 0, 300)
    theta_val = np.linspace(np.pi, np.pi - score * np.pi, 300)

    ax.plot(np.cos(theta_bg),  np.sin(theta_bg),
            lw=18, color='#2a2a4e', solid_capstyle='round')
    ax.plot(np.cos(theta_val), np.sin(theta_val),
            lw=18, color=color, solid_capstyle='round', alpha=0.9)

    ax.text(0, 0.25, f'{score:.3f}', ha='center', va='center',
            fontsize=30, fontweight='bold', color=color)
    ax.text(0, -0.1, grade(score), ha='center', va='center',
            fontsize=14, color=color, fontweight='bold')
    ax.text(0, -0.35, title, ha='center', va='center',
            fontsize=10, color='#aaaaaa')

    ax.set_xlim(-1.3, 1.3)
    ax.set_ylim(-0.5, 1.3)
    ax.axis('off')
    ax.set_facecolor('#1a1a2e')


fig, axes = plt.subplots(1, 2, figsize=(10, 4),
                          subplot_kw={'aspect': 'equal'})
fig.patch.set_facecolor('#1a1a2e')

draw_gauge(es_orig, 'Original Model\nEthical Score', axes[0])
draw_gauge(es_miti, 'Mitigated Model\nEthical Score', axes[1])

plt.tight_layout()
plt.show()

## 9. Conclusions

### What we found

The Logistic Regression model achieves strong predictive performance (~92% accuracy,
AUC ~0.97). The dominant feature is `cibil_score` — which makes intuitive sense
since it is specifically designed to measure credit risk.

On fairness, the results depend on how closely approval rates differ between
Graduate and Not Graduate applicants. The metrics tell us whether any gap is
within the acceptable threshold or crosses into discriminatory territory.

### Bias mitigation

Removing `education` from the feature set is a simple first step. It usually
closes some of the fairness gap with minimal accuracy cost. However, it is not
a complete solution — proxy features (income, assets) may still carry the
education signal indirectly.

### Limitations

- **Pre-processing mitigation only** — more advanced techniques like adversarial
  debiasing or reweighing could be explored.
- **Binary sensitive attribute** — we only look at Graduate vs Not Graduate.
  Real-world fairness often requires intersectional analysis.
- **Logistic Regression baseline** — a more powerful model (XGBoost, Random Forest)
  might perform better but could also encode bias more subtly.

### Next steps

1. Try reweighing or adversarial debiasing for stronger mitigation
2. Evaluate fairness across multiple sensitive attributes simultaneously
3. Run calibration analysis — is the model equally well calibrated across groups?
4. Consider fairness-aware model selection rather than post-hoc analysis